In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Geometry-V1-QK-E0 B2 thin operational handoff

Run this notebook once, top to bottom, only after separately authorizing model, GPU, Colab, and Drive use. It is a one-shot E0 representation/equivariance feasibility handoff: `science_denominator=0`, and it does not establish method, detector, or scientific success. Geometry recovers coordinates only.

The existing runner owns Q/K observation, 64-unit calculation, metrics, and packaging. This notebook never retries, switches layers, falls back, tunes, reads ZIP bytes, or calculates artifact hashes. Any failure stops the handoff.

In [ ]:
import json
import os
import pathlib
import subprocess
import sys
from datetime import datetime, timezone
from google.colab import userdata

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
BRANCH = 'Geometry-V1'
RUNNER_PATH = 'experiments/run_geometry_v1_qk_equivariance_operational.py'
SUCCESS_PREFIX = 'CEGWM_GEOMETRY_V1_QK_E0 '
FAILURE_PREFIX = 'CEGWM_GEOMETRY_V1_QK_E0_FAILURE '
MAX_CONTROL_BYTES = 1024
HANDOFF_FAILED = False
RUNNER_ATTEMPTED = False

def handoff_fail(stage, error):
    global HANDOFF_FAILED
    if HANDOFF_FAILED:
        return
    HANDOFF_FAILED = True
    print('CEGWM_GEOMETRY_V1_QK_E0_B2_FAILURE ' + json.dumps({
        'stage': stage, 'error_class': type(error).__name__
    }, sort_keys=True, separators=(',', ':')), flush=True)

def git_output(repo, *args):
    return subprocess.run(['git', *args], cwd=repo, check=True, capture_output=True, text=True).stdout.strip()


In [ ]:
if not HANDOFF_FAILED:
    try:
        session_utc = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        repo = pathlib.Path('/content') / ('geometry-v1-qk-e0-' + session_utc)
        if repo.exists():
            raise FileExistsError('fresh checkout path already exists')
        subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH, REPO_URL, str(repo)], check=True)
        execution_commit = git_output(repo, 'rev-parse', 'HEAD')
        checkout_branch = git_output(repo, 'branch', '--show-current')
        checkout_clean = git_output(repo, 'status', '--porcelain') == ''
        checkout_record = {'branch': checkout_branch, 'commit': execution_commit, 'clean': checkout_clean}
        print('CEGWM_GEOMETRY_V1_QK_E0_CHECKOUT ' + json.dumps(checkout_record, sort_keys=True))
        if checkout_branch != BRANCH or not checkout_clean:
            raise RuntimeError('fresh checkout identity differs')
        subprocess.run([sys.executable, '-m', 'pip', 'install', str(repo)], check=True)
    except BaseException as error:
        handoff_fail('checkout_or_install', error)


In [ ]:
# Fill this once with exactly two asymmetric RGB references and each one's
# four fixed attacked images. Hashes and source grids are user-supplied plan
# fields required by the existing runner; this notebook does not derive them.
ATTENTION_LAYER_PATHS = ['transformer_blocks.0.attn', 'transformer_blocks.23.attn']
TRANSFORMS = ('identity', 'd4', 'similarity', 'crop_rescale')
REFERENCES = []
# Each reference entry must contain:
# {'reference_id': 'r0', 'reference_path': '/content/drive/...png',
#  'reference_sha256': '<provided>', 'reference_source_grid': [height, width],
#  'attacks': {'identity': {'attacked_path': '...', 'attacked_sha256': '...',
#      'attacked_source_grid': [height, width], 'matched_h': [[...], [...], [...]],
#      'shuffled_h': [[...], [...], [...]}, ...}}

def build_e0_plan(references):
    if not isinstance(references, list) or len(references) != 2:
        raise ValueError('exactly two user-provided asymmetric references are required')
    pairs = []
    for reference in references:
        attacks = reference.get('attacks')
        if not isinstance(attacks, dict) or set(attacks) != set(TRANSFORMS):
            raise ValueError('each reference requires the fixed four-transform plan')
        for transform_label in TRANSFORMS:
            attack = attacks[transform_label]
            pairs.append({
                'reference_id': reference['reference_id'],
                'pair_id': reference['reference_id'] + '-' + transform_label,
                'transform_label': transform_label,
                'reference_path': reference['reference_path'],
                'reference_sha256': reference['reference_sha256'],
                'attacked_path': attack['attacked_path'],
                'attacked_sha256': attack['attacked_sha256'],
                'reference_source_grid': reference['reference_source_grid'],
                'attacked_source_grid': attack['attacked_source_grid'],
                # Known H is evaluation truth only; it never enters a detector API.
                'matched_h': attack['matched_h'],
                'shuffled_h': attack['shuffled_h'],
            })
    return {'schema': 'geometry-v1-qk-e0-plan-v1', 'attention_layer_paths': ATTENTION_LAYER_PATHS, 'pairs': pairs}

if not HANDOFF_FAILED:
    try:
        plan = build_e0_plan(REFERENCES)
        if len(plan['pairs']) != 8:
            raise RuntimeError('fixed eight-pair E0 plan was not formed')
        plan_path = pathlib.Path('/content') / ('geometry-v1-qk-e0-plan-' + session_utc + '.json')
        with plan_path.open('x', encoding='utf-8') as handle:
            json.dump(plan, handle, sort_keys=True, separators=(',', ':'))
    except BaseException as error:
        handoff_fail('evaluation_plan', error)


In [ ]:
hf_token = ''
runner_env = None
control_read = None
control_write = None
if not HANDOFF_FAILED:
    try:
        if RUNNER_ATTEMPTED:
            raise RuntimeError('runner was already attempted')
        RUNNER_ATTEMPTED = True
        hf_token = userdata.get('HF_TOKEN')
        if not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError('HF_TOKEN Colab Secret is required')
        run_id = 'geometry-v1-qk-e0-' + execution_commit[:12]
        drive_root = pathlib.Path('/content/drive/MyDrive/CEG-WM/Geometry-V1/E0')
        run_dir = drive_root / ('Geometry-V1-QK-E0-' + execution_commit[:12] + '-' + session_utc)
        if run_dir.exists():
            raise FileExistsError('create-only Drive run directory already exists')
        runner_env = {name: value for name, value in os.environ.items() if 'TOKEN' not in name.upper()}
        runner_env['HF_TOKEN'] = hf_token
        hf_token = ''
        control_read, control_write = os.pipe()
        command = [sys.executable, RUNNER_PATH, '--plan', str(plan_path), '--repo-root', str(repo),
                   '--expected-exact', execution_commit, '--output-root', str(run_dir),
                   '--control-fd', str(control_write)]
        process = subprocess.Popen(command, cwd=repo, env=runner_env, pass_fds=(control_write,),
                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        os.close(control_write); control_write = None
        runner_rc = process.wait(timeout=7200)
        line = os.read(control_read, MAX_CONTROL_BYTES + 1)
        if len(line) > MAX_CONTROL_BYTES or not line.endswith(b'\n'):
            raise RuntimeError('invalid compact control receipt')
        text = line.decode('utf-8', 'strict').strip()
        if text.startswith(SUCCESS_PREFIX):
            receipt_status, receipt = 'success', json.loads(text[len(SUCCESS_PREFIX):])
        elif text.startswith(FAILURE_PREFIX):
            receipt_status, receipt = 'failure', json.loads(text[len(FAILURE_PREFIX):])
        else:
            raise RuntimeError('unexpected control prefix')
        if receipt.get('run_id') != run_id or receipt.get('status') != receipt_status:
            raise RuntimeError('control receipt identity differs')
        print('CEGWM_GEOMETRY_V1_QK_E0_TERMINAL ' + json.dumps({
            'branch': checkout_branch, 'commit': execution_commit, 'run_directory': str(run_dir),
            'runner_rc': runner_rc, 'receipt_status': receipt_status,
            'artifact_status': receipt.get('artifact_status'),
            'failure_point': receipt.get('failure_point'), 'science_denominator': 0
        }, sort_keys=True))
        if receipt_status != 'success':
            raise RuntimeError('runner reported failure; handoff stops')
    except BaseException as error:
        handoff_fail('runner', error)
    finally:
        hf_token = ''
        if runner_env is not None:
            runner_env.pop('HF_TOKEN', None)
        for fd in (control_read, control_write):
            if fd is not None:
                os.close(fd)


## Stop boundary

The runner's terminal control receipt is the only result displayed here. This notebook does not inspect ZIP contents, calculate artifact hashes, retry a failure, or make a route, detector, method, or scientific decision. A real B2 execution requires separate authorization bound to the then-current exact and the model/GPU/Colab/Drive action.